In [29]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.path.join("..","data","processed")
IN_FILE  = os.path.join(DATA_DIR, "panel_BALANCED_MAIN_JanJul_2020_2024_2025_AB_v1.csv")

df = pd.read_csv(IN_FILE, parse_dates=["date"], engine="pyarrow")

df.head()



,date,station_code,period_window,year,month,dow,PM2.5,NO2,CO,O3,NO,NOX_final,PM2.5_AB_imputed,NO2_AB_imputed,CO_AB_imputed,O3_AB_imputed,NO_AB_imputed,NOX_final_AB_imputed
0,2024-01-01 00:00:00,CE,baseline_JanJul2024,2024,1,0,NaN,20.55,1.91,19.5,4.15,26.6,True,True,True,True,True,True
1,2024-01-01 01:00:00,CE,baseline_JanJul2024,2024,1,0,NaN,35.30,2.53,14.0,5.70,41.0,True,False,False,False,False,False
2,2024-01-01 02:00:00,CE,baseline_JanJul2024,2024,1,0,NaN,24.90,2.16,19.0,4.00,28.9,False,False,False,False,False,False
3,2024-01-01 03:00:00,CE,baseline_JanJul2024,2024,1,0,NaN,21.00,1.92,20.0,3.50,24.5,False,False,False,False,False,False
4,2024-01-01 04:00:00,CE,baseline_JanJul2024,2024,1,0,NaN,17.50,1.74,25.0,3.50,21.0,False,False,False,False,False,False


In [30]:
print("Hola Mundo!")


Hola Mundo!


In [31]:
# Convertir df a formato largo: datetime, station, pollutant, value
pollutant_cols = ['PM2.5','NO2','CO','O3','NO','NOX_final']
present = [c for c in pollutant_cols if c in df.columns]

long_df = (
    df.rename(columns={'date': 'datetime', 'station_code': 'station'})
      .melt(id_vars=['datetime', 'station'],
            value_vars=present,
            var_name='pollutant',
            value_name='value')
)

# Ignorar filas con NaN en value
long_df = long_df.dropna(subset=['value'])

long_df.head()


,datetime,station,pollutant,value
5065,2024-01-01 00:00:00,NE,PM2.5,999.0
5066,2024-01-01 01:00:00,NE,PM2.5,464.0
5067,2024-01-01 02:00:00,NE,PM2.5,690.0
5068,2024-01-01 03:00:00,NE,PM2.5,729.0
5069,2024-01-01 04:00:00,NE,PM2.5,557.0


In [32]:
# Métricas mensuales por estación y contaminante usando AUC (trapecio, paso 1h)
import numpy as np
import pandas as pd
import os

ldf = long_df.copy()
ldf['datetime'] = pd.to_datetime(ldf['datetime'])
ldf['month'] = ldf['datetime'].dt.to_period('M').dt.to_timestamp()

def _monthly_metrics(g):
    s = g.set_index('datetime').sort_index().asfreq('1H')
    v = s['value']
    valid_hours = int(v.notna().sum())
    m = v.notna() & v.shift(-1).notna()
    auc = float((0.5 * (v[m] + v.shift(-1)[m])).sum())
    mean_monthly = (auc / valid_hours) if valid_hours > 0 else np.nan
    arr = v.to_numpy()
    p50 = float(np.nanpercentile(arr, 50)) if valid_hours > 0 else np.nan
    p90 = float(np.nanpercentile(arr, 90)) if valid_hours > 0 else np.nan
    daily_max = s['value'].resample('1D').max(min_count=1)
    max_diario = float(daily_max.max()) if daily_max.notna().any() else np.nan
    return pd.Series({
        'auc': auc,
        'mean': mean_monthly,
        'p50': p50,
        'p90': p90,
        'max_diario': max_diario,
        'valid_hours': valid_hours,
    })

metrics_monthly = (
    ldf.groupby(['station','pollutant','month'], as_index=False)
       .apply(_monthly_metrics)
       .reset_index(drop=True)
)

# Guardar CSV
out_dir = os.path.join('..','reports')
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, 'metrics_monthly.csv')
metrics_monthly.to_csv(out_file, index=False)
out_file, metrics_monthly.head()


C:\Users\Val\AppData\Local\Temp\ipykernel_1204\625224768.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s = g.set_index('datetime').sort_index().asfreq('1H')
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\625224768.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_monthly_metrics)


('..\\reports\\metrics_monthly.csv',
   station pollutant      month        auc      mean    p50    p90  max_diario  \
 0      CE        CO 2020-01-01  1916.3700  2.575766  2.545  3.777        5.34   
 1      CE        CO 2020-02-01  1789.2250  2.570726  2.540  3.035        4.55   
 2      CE        CO 2020-03-01  2421.8000  3.255108  3.260  3.730        4.53   
 3      CE        CO 2020-04-01   630.7875  0.876094  0.810  1.190        3.54   
 4      CE        CO 2020-05-01  1072.4275  1.441435  1.460  1.700        2.36   
 
    valid_hours  
 0        744.0  
 1        696.0  
 2        744.0  
 3        720.0  
 4        744.0  )

In [33]:
# IAQI diario para PM2.5 (24h) y O3 (máx 8h móvil) con reglas US EPA
# Supuesto: O3 en ppb -> convertir a ppm dividiendo por 1000 antes de IAQI
import numpy as np
import pandas as pd
import os

ldf = long_df.copy()
ldf['datetime'] = pd.to_datetime(ldf['datetime'])
ldf = ldf.set_index('datetime').sort_index()

def truncate(x, decimals):
    factor = 10 ** decimals
    return np.floor(x * factor) / factor

# Breakpoints EPA
PM25_BREAKPOINTS = [
    (0.0, 12.0, 0, 50),
    (12.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200),
    (150.5, 250.4, 201, 300),
    (250.5, 350.4, 301, 400),
    (350.5, 500.4, 401, 500),
]

O3_8H_BREAKPOINTS = [
    (0.000, 0.054, 0, 50),
    (0.055, 0.070, 51, 100),
    (0.071, 0.085, 101, 150),
    (0.086, 0.105, 151, 200),
    (0.106, 0.200, 201, 300),
]

CO_8H_BREAKPOINTS = [
    (0.0, 4.4, 0, 50),
    (4.5, 9.4, 51, 100),
    (9.5, 12.4, 101, 150),
    (12.5, 15.4, 151, 200),
    (15.5, 30.4, 201, 300),
    (30.5, 40.4, 301, 400),
    (40.5, 50.4, 401, 500),
]

NO2_1H_BREAKPOINTS = [
    (0, 53, 0, 50),
    (54, 100, 51, 100),
    (101, 360, 101, 150),
    (361, 649, 151, 200),
    (650, 1249, 201, 300),
    (1250, 1649, 301, 400),
    (1650, 2049, 401, 500),
]

def iaqi_from_bp(c, bps):
    if pd.isna(c):
        return np.nan
    for Clow, Chigh, Ilow, Ihigh in bps:
        if c <= Chigh:
            return (Ihigh - Ilow) / (Chigh - Clow) * (c - Clow) + Ilow
    Clow, Chigh, Ilow, Ihigh = bps[-1]
    return (Ihigh - Ilow) / (Chigh - Clow) * (c - Clow) + Ilow

# ---- PM2.5: promedio diario 24h, requiere >= 18 horas válidas (75%) ----
pm = ldf[ldf['pollutant'] == 'PM2.5'].copy()
pm_results = []
for station, g in pm.groupby('station'):
    s = g['value'].asfreq('1H')
    daily_count = s.resample('1D').count()
    daily_mean = s.resample('1D').mean()
    daily_mean = daily_mean.where(daily_count >= 18)
    conc_24h = truncate(daily_mean, 1)
    iaqi_pm25 = conc_24h.apply(lambda x: iaqi_from_bp(x, PM25_BREAKPOINTS))
    iaqi_pm25 = iaqi_pm25.round(0)
    pm_results.append(pd.DataFrame({
        'station': station,
        'date': conc_24h.index,
        'iaqi_pm25': iaqi_pm25
    }))
pm25_daily = pd.concat(pm_results, ignore_index=True) if pm_results else pd.DataFrame(columns=['station','date','iaqi_pm25'])

# ---- O3: máximo diario de medias móviles 8h, cada 8h requiere >= 6 datos (75%) ----
o3 = ldf[ldf['pollutant'] == 'O3'].copy()
o3_results = []
for station, g in o3.groupby('station'):
    s = g['value'].asfreq('1H')
    s_ppm = s / 1000.0  # ppb -> ppm
    o3_8h = s_ppm.rolling(window=8, min_periods=6).mean()
    o3_8h = truncate(o3_8h, 3)
    daily_max8h = o3_8h.resample('1D').max()
    iaqi_o3 = daily_max8h.apply(lambda x: iaqi_from_bp(x, O3_8H_BREAKPOINTS))
    iaqi_o3 = iaqi_o3.round(0)
    o3_results.append(pd.DataFrame({
        'station': station,
        'date': daily_max8h.index,
        'iaqi_o3': iaqi_o3
    }))
o3_daily = pd.concat(o3_results, ignore_index=True) if o3_results else pd.DataFrame(columns=['station','date','iaqi_o3'])

# ---- CO: máximo diario de medias móviles 8h, cada 8h requiere >= 6 datos (75%)) ----
co = ldf[ldf['pollutant'] == 'CO'].copy()
co_results = []
for station, g in co.groupby('station'):
    s = g['value'].asfreq('1H')
    co_8h = s.rolling(window=8, min_periods=6).mean()
    co_8h = truncate(co_8h, 1)  # ppm truncado a 0.1
    daily_max8h = co_8h.resample('1D').max()
    iaqi_co = daily_max8h.apply(lambda x: iaqi_from_bp(x, CO_8H_BREAKPOINTS))
    iaqi_co = iaqi_co.round(0)
    co_results.append(pd.DataFrame({
        'station': station,
        'date': daily_max8h.index,
        'iaqi_co': iaqi_co
    }))
co_daily = pd.concat(co_results, ignore_index=True) if co_results else pd.DataFrame(columns=['station','date','iaqi_co'])

# ---- NO2: máximo diario 1h, requiere >= 18 horas válidas en el día ----
no2 = ldf[ldf['pollutant'] == 'NO2'].copy()
no2_results = []
for station, g in no2.groupby('station'):
    s = g['value'].asfreq('1H')  # ppb
    daily_count = s.resample('1D').count()
    max_1h = s.resample('1D').max()
    max_1h = max_1h.where(daily_count >= 18)
    conc_1h = truncate(max_1h, 0)  # ppb truncado a entero
    iaqi_no2 = conc_1h.apply(lambda x: iaqi_from_bp(x, NO2_1H_BREAKPOINTS))
    iaqi_no2 = iaqi_no2.round(0)
    no2_results.append(pd.DataFrame({
        'station': station,
        'date': conc_1h.index,
        'iaqi_no2': iaqi_no2
    }))
no2_daily = pd.concat(no2_results, ignore_index=True) if no2_results else pd.DataFrame(columns=['station','date','iaqi_no2'])

# ---- AQI final del día: máximo IAQI disponible (PM2.5, O3, CO, NO2) ----
aqi_daily = (pm25_daily
             .merge(o3_daily, on=['station','date'], how='outer')
             .merge(co_daily, on=['station','date'], how='outer')
             .merge(no2_daily, on=['station','date'], how='outer'))
aqi_daily['aqi'] = aqi_daily[['iaqi_pm25','iaqi_o3','iaqi_co','iaqi_no2']].max(axis=1, skipna=True)
aqi_daily = aqi_daily.sort_values(['station','date']).reset_index(drop=True)

# Guardar CSV
out_dir = os.path.join('..','reports')
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, 'aqi_daily.csv')
aqi_daily.to_csv(out_file, index=False)
out_file, aqi_daily.head()


C:\Users\Val\AppData\Local\Temp\ipykernel_1204\685030686.py:67: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s = g['value'].asfreq('1H')
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\685030686.py:85: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s = g['value'].asfreq('1H')
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\685030686.py:103: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s = g['value'].asfreq('1H')
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\685030686.py:120: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  s = g['value'].asfreq('1H')  # ppb


('..\\reports\\aqi_daily.csv',
   station       date  iaqi_pm25  iaqi_o3  iaqi_co  iaqi_no2   aqi
 0      CE 2020-01-01        NaN     23.0     44.0       9.0  44.0
 1      CE 2020-01-02        NaN     24.0     41.0      15.0  41.0
 2      CE 2020-01-03        NaN     44.0     41.0      14.0  44.0
 3      CE 2020-01-04        NaN     40.0     35.0      16.0  40.0
 4      CE 2020-01-05        NaN     48.0     36.0      19.0  48.0)

In [34]:
# Cargar metrics_monthly.csv y graficar AUC y media (mediana e IQR) para 2020, 2024 y 2025 (enero-julio), por contaminante
# Usa un conjunto fijo de estaciones presentes en los tres años por contaminante y anota valores en las barras
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt

METRICS_FILE = os.path.join('..','reports','metrics_monthly.csv')
m = pd.read_csv(METRICS_FILE, parse_dates=['month'])
m['year'] = m['month'].dt.year
m['mon'] = m['month'].dt.month
m = m[m['year'].isin([2020, 2024, 2025])]
m = m[m['mon'].between(1,7)]  # enero-julio

months = list(range(1,8))
labels = ['Ene','Feb','Mar','Abr','May','Jun','Jul']
years = [2020, 2024, 2025]
colors = {2020:'#4e79a7', 2024:'#f28e2b', 2025:'#59a14f'}
out_dir = os.path.join('..','reports')
os.makedirs(out_dir, exist_ok=True)

saved = []
for pol in sorted(m['pollutant'].unique()):
    mp = m[m['pollutant']==pol].copy()
    # Estaciones presentes en los 3 años (para este contaminante)
    sets = [set(mp[mp['year']==y]['station'].unique()) for y in years]
    fixed = set.intersection(*sets) if all(len(s)>0 for s in sets) else set()
    mp = mp[mp['station'].isin(fixed)].copy()
    if mp.empty:
        continue

    # Agregación por año/mes: mediana e IQR entre estaciones
    def _agg(gr):
        auc_vals = gr['auc'].astype(float).to_numpy()
        mean_vals = gr['mean'].astype(float).to_numpy()
        med_auc = float(np.nanmedian(auc_vals)) if auc_vals.size else np.nan
        q25_auc = float(np.nanquantile(auc_vals, 0.25)) if auc_vals.size else np.nan
        q75_auc = float(np.nanquantile(auc_vals, 0.75)) if auc_vals.size else np.nan
        med_mean = float(np.nanmedian(mean_vals)) if mean_vals.size else np.nan
        q25_mean = float(np.nanquantile(mean_vals, 0.25)) if mean_vals.size else np.nan
        q75_mean = float(np.nanquantile(mean_vals, 0.75)) if mean_vals.size else np.nan
        return pd.Series({'auc_median': med_auc, 'auc_q25': q25_auc, 'auc_q75': q75_auc,
                             'mean_median': med_mean, 'mean_q25': q25_mean, 'mean_q75': q75_mean})
    agg = (mp.groupby(['year','mon'], as_index=False)
             .apply(_agg)
             .reset_index(drop=True))

    fig, axes = plt.subplots(1,2, figsize=(14,4), sharex=True)
    width = 0.25
    x = np.arange(len(months))
    # AUC mediana con barras de error (IQR) + anotaciones
    ax = axes[0]
    for i,y in enumerate(years):
        dfy = agg[agg['year']==y].set_index('mon').reindex(months)
        med = dfy['auc_median'].values
        low = (dfy['auc_median'] - dfy['auc_q25']).values
        upp = (dfy['auc_q75'] - dfy['auc_median']).values
        yerr = np.vstack([low, upp])
        rects = ax.bar(x + (i-1)*width, med, width=width, label=str(y), color=colors[y], yerr=yerr, capsize=3)
    ax.set_title(f'AUC mensual (mediana, IQR) — {pol}')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('AUC')
    ax.legend(title='Año')
    ax.grid(axis='y', alpha=0.3)
    # Media mediana con IQR + anotaciones
    ax = axes[1]
    for i,y in enumerate(years):
        dfy = agg[agg['year']==y].set_index('mon').reindex(months)
        med = dfy['mean_median'].values
        low = (dfy['mean_median'] - dfy['mean_q25']).values
        upp = (dfy['mean_q75'] - dfy['mean_median']).values
        yerr = np.vstack([low, upp])
        rects = ax.bar(x + (i-1)*width, med, width=width, label=str(y), color=colors[y], yerr=yerr, capsize=3)
    ax.set_title(f'Media mensual (mediana, IQR) — {pol}')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Concentración')
    ax.legend(title='Año')
    ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'Comparación 2020 vs 2024 vs 2025 (Ene–Jul) — {pol}', y=1.03, fontsize=12)
    fig.tight_layout()
    fname = os.path.join(out_dir, f'metrics_monthly_{pol}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    saved.append(fname)

saved


C:\Users\Val\AppData\Local\Temp\ipykernel_1204\1600419392.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\1600419392.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\Val\AppData\Local\Temp\ipykernel_1204\1600419392.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in

['..\\reports\\metrics_monthly_CO.png',
 '..\\reports\\metrics_monthly_NO.png',
 '..\\reports\\metrics_monthly_NO2.png',
 '..\\reports\\metrics_monthly_NOX_final.png',
 '..\\reports\\metrics_monthly_O3.png',
 '..\\reports\\metrics_monthly_PM2.5.png']

In [35]:
# AQI de ciudad por día (máximo entre estaciones), categorización EPA y media móvil 7d; gráfico 2020/2024/2025
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates

# Usar aqi_daily en memoria si existe; si no, cargar CSV
try:
    aqi_df = aqi_daily.copy()
except NameError:
    AQI_FILE = os.path.join('..','reports','aqi_daily.csv')
    aqi_df = pd.read_csv(AQI_FILE, parse_dates=['date'])

aqi_df['date'] = pd.to_datetime(aqi_df['date'])
aqi_df = aqi_df.dropna(subset=['aqi'])
# AQI ciudad: máximo por fecha
city = (aqi_df.groupby('date', as_index=False)['aqi'].max().sort_values('date'))
city['year'] = city['date'].dt.year
# Categorías EPA
def cat(a):
    if pd.isna(a): return np.nan
    a = float(a)
    if a <= 50: return 'Good'
    if a <= 100: return 'Moderate'
    if a <= 150: return 'USG'
    if a <= 200: return 'Unhealthy'
    if a <= 300: return 'Very Unhealthy'
    return 'Hazardous'
city['category'] = city['aqi'].apply(cat)
# Media móvil 7d por serie completa y también por año para trazado limpio
city = city.set_index('date').sort_index()
city['aqi_ma7'] = city['aqi'].rolling(window=7, min_periods=1).mean()
city['year'] = city.index.year

years = [2020, 2024, 2025]
colors = {2020:'#4e79a7', 2024:'#f28e2b', 2025:'#59a14f'}
# Guardar CSV con AQI ciudad (date, aqi, category, aqi_ma7)
reports_dir = os.path.join('..','reports')
os.makedirs(reports_dir, exist_ok=True)
city_out = city.reset_index()[['date','aqi','category','aqi_ma7']]
city_csv = os.path.join(reports_dir, 'aqi_city_daily.csv')
city_out.to_csv(city_csv, index=False)

out_dir = os.path.join('..','figs')
os.makedirs(out_dir, exist_ok=True)

files = []
for y in years:
    sub = city[city['year']==y]
    # Restringir a Ene–Jul para comparabilidad
    sub = sub[(sub.index.month>=1) & (sub.index.month<=7)]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(12,4))
    # Bandas de fondo por categorías EPA
    bands = [(0,50,'Good','#00e400'), (50,100,'Moderate','#ffff00'), (100,150,'USG','#ff7e00'), (150,200,'Unhealthy','#ff0000'), (200,300,'Very Unhealthy','#8f3f97'), (300,500,'Hazardous','#7e0023')]
    for lo, hi, name, color in bands:
        ax.axhspan(lo, hi, color=color, alpha=0.08, zorder=0)
    # Límites Y y líneas de referencia
    ymax = float(np.nanmax(sub['aqi'].values)) if len(sub) else 0.0
    ax.set_ylim(0, max(350, min(500, ymax + 25)))
    for t in [50, 100, 150, 200, 300]:
        ax.axhline(t, color='#444', linewidth=0.8, linestyle='--', alpha=0.6, zorder=1)
    # Series
    ax.plot(sub.index, sub['aqi'], color=colors[y], alpha=0.25, linewidth=1, label='diario')
    ax.plot(sub.index, sub['aqi_ma7'], color=colors[y], alpha=0.95, linewidth=2, label='media 7d')
    # Formato de eje X por meses Ene–Jul
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1,8)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(pd.Timestamp(f'{y}-01-01'), pd.Timestamp(f'{y}-07-31'))
    ax.set_title(f'AQI diario de ciudad y media móvil 7 días — {y} (Ene–Jul)')
    ax.set_ylabel('AQI')
    ax.grid(alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fname = os.path.join(out_dir, f'AQI_timeline_city_{y}.png')
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    files.append(fname)
(city_csv, files)


('..\\reports\\aqi_city_daily.csv',
 ['..\\figs\\AQI_timeline_city_2020.png',
  '..\\figs\\AQI_timeline_city_2024.png',
  '..\\figs\\AQI_timeline_city_2025.png'])

In [36]:
# Barras por categorías (días por año) usando AQI de ciudad; también porcentaje.
# Se restringe a Ene–Jul para comparabilidad entre 2020/2024/2025.
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt

try:
    city_df = city.copy()  # de la celda anterior
    city_df = city_df.reset_index()
except Exception:
    CITY_CSV = os.path.join('..','reports','aqi_city_daily.csv')
    city_df = pd.read_csv(CITY_CSV, parse_dates=['date'])

city_df['year'] = city_df['date'].dt.year
city_df['mon'] = city_df['date'].dt.month
city_df = city_df[city_df['mon'].between(1,7)]

years = [y for y in [2020, 2024, 2025] if y in set(city_df['year'])]
cats = ['Good','Moderate','USG','Unhealthy','Very Unhealthy','Hazardous']
cat_colors = {'Good':'#00e400','Moderate':'#ffff00','USG':'#ff7e00','Unhealthy':'#ff0000','Very Unhealthy':'#8f3f97','Hazardous':'#7e0023'}

cnt = (city_df.groupby(['year','category']).size().rename('days').reset_index())
cnt = cnt[cnt['year'].isin(years)]
pivot = cnt.pivot(index='year', columns='category', values='days').reindex(years).fillna(0)
pivot = pivot.reindex(columns=cats).fillna(0)

out_dir = os.path.join('..','figs')
os.makedirs(out_dir, exist_ok=True)
# Stacked counts
fig, ax = plt.subplots(figsize=(10,5))
bottom = np.zeros(len(pivot))
x = np.arange(len(pivot.index))
for c in cats:
    vals = pivot[c].values
    ax.bar(x, vals, bottom=bottom, label=c, color=cat_colors[c])
    bottom = bottom + vals
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in pivot.index])
ax.set_ylabel('Días (Ene–Jul)')
ax.set_title('Días por categoría AQI — Ciudad (Ene–Jul)')
ax.legend(ncol=3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fname1 = os.path.join(out_dir, 'AQI_days_by_category_city.png')
fig.savefig(fname1, dpi=150, bbox_inches='tight')
plt.close(fig)

# Normalized percentages
row_sums = pivot.sum(axis=1).replace(0, np.nan)
pct = pivot.divide(row_sums, axis=0) * 100
fig, ax = plt.subplots(figsize=(10,5))
bottom = np.zeros(len(pct))
for c in cats:
    vals = pct[c].values
    ax.bar(x, vals, bottom=bottom, label=c, color=cat_colors[c])
    bottom = bottom + np.nan_to_num(vals)
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in pct.index])
ax.set_ylabel('% de días (Ene–Jul)')
ax.set_ylim(0, 100)
ax.set_title('Distribución porcentual por categoría AQI — Ciudad (Ene–Jul)')
ax.legend(ncol=3)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fname2 = os.path.join(out_dir, 'AQI_days_by_category_city_pct.png')
fig.savefig(fname2, dpi=150, bbox_inches='tight')
plt.close(fig)
(fname1, fname2)


('..\\figs\\AQI_days_by_category_city.png',
 '..\\figs\\AQI_days_by_category_city_pct.png')